# NER Inference End-to-End

**Input:** Văn bản bệnh án thô (paste vào TEXT variable)

**Output:** JSON với 5 loại thực thể

**Kaggle Datasets cần add:**
1. NER encoder weights (từ notebook 1)
2. icd10_vi_full.csv
3. rxnorm_merged.csv + inn_usan.csv

**QUAN TRỌNG:** Chỉ chạy cell 1 (cài đặt) và cell 2 (nhập text), các cell còn lại tự động chạy!

In [ ]:
# Cell 1: Cài đặt và verify
!pip install -q transformers pyvi sentence-transformers vllm pandas

import os
import sys

# Clone repo
REPO_URL = "https://github.com/Khanhhh239/fakeer.git"
REPO_DIR = "fakeer"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.append('src')

print("✓ Setup complete!")

In [ ]:
# Cell 2: NHẬP TEXT Ở ĐÂY - ô duy nhất bạn cần sửa!

TEXT = """
PASTE BỆNH ÁN VÀO ĐÂY
""".strip()

print(f"Text length: {len(TEXT)} characters")
print(f"Preview: {TEXT[:200]}...")

In [ ]:
# Cell 3: NHÁNH B - Xét nghiệm (luật regex)
from branch_b_lab_tests import extract_lab_pairs

print("Running Branch B (Lab Tests)...")
lab_entities = extract_lab_pairs(TEXT)

print(f"\n✓ Found {len(lab_entities)} lab test entities")
for ent in lab_entities[:10]:  # Show first 10
    print(f"  [{ent['type']}] '{ent['text']}'")

In [ ]:
# Cell 4: NHÁNH C - Thuốc (từ điển RxNorm)
from branch_c_drugs import DrugMatcher

# TODO: Update paths to your Kaggle Datasets
RXNORM_PATH = '/kaggle/input/medical-kb/rxnorm_merged.csv'
INN_USAN_PATH = '/kaggle/input/medical-kb/inn_usan.csv'

print("Loading drug matcher...")
drug_matcher = DrugMatcher(
    rxnorm_path=RXNORM_PATH,
    inn_usan_path=INN_USAN_PATH
)

# Add common Vietnamese drugs not in RxNorm
vietnamese_drugs = [
    "Medrol", "Zestril", "Vastarel", "Nitralmyl",
    # Add more as needed
]
drug_matcher.add_custom_drugs(vietnamese_drugs)

print("\nRunning Branch C (Drugs)...")
drug_entities = drug_matcher.extract_drugs(TEXT)

print(f"\n✓ Found {len(drug_entities)} drug entities")
for ent in drug_entities[:10]:
    print(f"  [THUỐC] '{ent['text']}'")

In [ ]:
# Cell 5: NHÁNH A Bước 1 - Tách từ + ánh xạ offset (CRITICAL!)
from utils.text_alignment import segment_with_map

print("Segmenting text with offset mapping...")
words, spans, ok = segment_with_map(TEXT)

# CRITICAL: Phải assert ok!
assert ok, "⚠️ ÁNH XẠ OFFSET HỎNG - DỪNG NGAY!"

print(f"✓ Segmented into {len(words)} words")
print(f"✓ Offset mapping OK")
print(f"\nFirst 10 words with spans:")
for i in range(min(10, len(words))):
    start, end = spans[i]
    original = TEXT[start:end]
    print(f"  {i}: '{words[i]}' -> ({start}, {end}) = '{original}'")
    assert words[i].replace('_', '') == original.replace(' ', ''), f"Mismatch at {i}!"

In [ ]:
# Cell 6: NHÁNH A Bước 2 - Encoder BIO -> span SYM_DIS
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
import numpy as np

# TODO: Update path to your trained model
MODEL_PATH = '/kaggle/input/ner-encoder-weights/ner_encoder'

print("Loading NER encoder...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

# Load label map
import json
with open(f'{MODEL_PATH}/label_map.json') as f:
    label_map = json.load(f)
    id2label = {int(k): v for k, v in label_map['id2label'].items()}

print(f"✓ Model loaded on {device}")

# Tokenize
print("\nTokenizing...")
inputs = tokenizer(
    words,
    is_split_into_words=True,
    return_tensors='pt',
    truncation=True,
    max_length=512
).to(device)

# Inference
print("Running inference...")
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)[0].cpu().numpy()
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()

# Map back to words
word_ids = inputs.word_ids()
word_predictions = []
word_probs = []

for word_idx in range(len(words)):
    # Find first token of this word
    token_indices = [i for i, wid in enumerate(word_ids) if wid == word_idx]
    if token_indices:
        first_token = token_indices[0]
        word_predictions.append(id2label[predictions[first_token]])
        word_probs.append(probs[first_token])
    else:
        word_predictions.append('O')
        word_probs.append(np.array([1.0, 0.0, 0.0]))  # All O

print(f"✓ Predicted {len(word_predictions)} labels")

# Extract SYM_DIS spans
sym_dis_spans = []
current_span = None

for i, (word, label, prob) in enumerate(zip(words, word_predictions, word_probs)):
    if label == 'B-SYM_DIS':
        # Start new span
        if current_span:
            sym_dis_spans.append(current_span)
        current_span = {
            'word_start': i,
            'word_end': i,
            'probs': [prob]
        }
    elif label == 'I-SYM_DIS' and current_span:
        # Continue span
        current_span['word_end'] = i
        current_span['probs'].append(prob)
    else:
        # End span
        if current_span:
            sym_dis_spans.append(current_span)
            current_span = None

if current_span:
    sym_dis_spans.append(current_span)

# Convert to entities with offset mapping
encoder_entities = []
for span_info in sym_dis_spans:
    word_start = span_info['word_start']
    word_end = span_info['word_end']
    
    # Get character offsets
    char_start = spans[word_start][0]
    char_end = spans[word_end][1]
    
    # Get text from ORIGINAL
    span_text = TEXT[char_start:char_end]
    
    # Average probability
    avg_prob = np.mean([p[1] for p in span_info['probs']])  # Index 1 = B-SYM_DIS
    
    encoder_entities.append({
        'text': span_text,
        'type': 'SYM_DIS',  # Will be refined by cascade
        'start': char_start,
        'end': char_end,
        'score': float(avg_prob),
        'source': 'encoder'
    })

print(f"\n✓ Extracted {len(encoder_entities)} SYM_DIS spans")
for ent in encoder_entities[:10]:
    print(f"  [SYM_DIS] '{ent['text']}' (score: {ent['score']:.2f})")

In [ ]:
# Cell 7: NHÁNH A Bước 3 - Thác tách TRIỆU_CHỨNG / CHẨN_ĐOÁN
from cascade_classifier import CascadeClassifier

# TODO: Update path
ICD_PATH = '/kaggle/input/medical-kb/icd10_vi_full.csv'

print("Loading cascade classifier...")
cascade = CascadeClassifier(
    icd_path=ICD_PATH,
    embedding_model="AITeamVN/Vietnamese_Embedding",
    threshold=0.93
)

print("\nClassifying SYM_DIS spans...")
# Add context for each span
for ent in encoder_entities:
    # Simple context: surrounding ±50 chars
    start = max(0, ent['start'] - 50)
    end = min(len(TEXT), ent['end'] + 50)
    ent['context'] = TEXT[start:end]

# Classify (without LLM for now - tier 3 will default to TRIỆU_CHỨNG)
classified_entities = cascade.classify_batch(encoder_entities, llm_classifier=None)

# Statistics
tier_stats = {}
for ent in classified_entities:
    tier = ent.get('tier', 'unknown')
    tier_stats[tier] = tier_stats.get(tier, 0) + 1

print(f"\n✓ Classification complete!")
print(f"Tier statistics:")
for tier, count in tier_stats.items():
    print(f"  {tier}: {count}")

# Update source
for ent in classified_entities:
    tier = ent.get('tier', '')
    if 'kb' in tier:
        ent['source'] = 'encoder+kb'
    elif 'llm' in tier:
        ent['source'] = 'encoder+llm'

print(f"\nSample results:")
for ent in classified_entities[:10]:
    print(f"  [{ent['type']}] '{ent['text']}' (tier: {ent['tier']})")

In [ ]:
# Cell 8: Hợp nhất và khử chồng lấn
from utils.overlap_resolver import merge_entities_from_branches

print("Merging all entities...")
print(f"  Encoder entities: {len(classified_entities)}")
print(f"  Lab test entities: {len(lab_entities)}")
print(f"  Drug entities: {len(drug_entities)}")

# Combine rule-based entities
rule_entities = lab_entities + drug_entities

# Merge and resolve overlaps
final_entities = merge_entities_from_branches(
    encoder_entities=classified_entities,
    rule_entities=rule_entities,
    llm_entities=None
)

print(f"\n✓ Final entities: {len(final_entities)}")

# Type distribution
type_counts = {}
for ent in final_entities:
    t = ent['type']
    type_counts[t] = type_counts.get(t, 0) + 1

print(f"\nType distribution:")
for t, count in sorted(type_counts.items()):
    print(f"  {t}: {count}")

In [ ]:
# Cell 9: Checklist nghiệm thu (BẮT BUỘC)

print("="*60)
print("RUNNING VALIDATION CHECKLIST")
print("="*60)

VALID_TYPES = {"TRIỆU_CHỨNG", "CHẨN_ĐOÁN", "THUỐC", "TÊN_XÉT_NGHIỆM", "KẾT_QUẢ_XÉT_NGHIỆM"}

errors = []

# 1. Nguyên văn
for i, ent in enumerate(final_entities):
    if ent['text'] != TEXT[ent['start']:ent['end']]:
        errors.append(f"Entity {i}: Span KHÔNG nguyên văn!")

# 2. Không chồng lấn
sorted_ents = sorted(final_entities, key=lambda x: x['start'])
for i in range(len(sorted_ents) - 1):
    if sorted_ents[i]['end'] > sorted_ents[i + 1]['start']:
        errors.append(f"Entities {i} and {i+1}: CHỒNG LẤN!")

# 3. Type hợp lệ
for i, ent in enumerate(final_entities):
    if ent['type'] not in VALID_TYPES:
        errors.append(f"Entity {i}: Type LẠ: {ent['type']}")

# 4. Không rỗng
for i, ent in enumerate(final_entities):
    if not ent['text'].strip():
        errors.append(f"Entity {i}: Span RỖNG!")

# 5. Offset hợp lệ
for i, ent in enumerate(final_entities):
    if not (0 <= ent['start'] < ent['end'] <= len(TEXT)):
        errors.append(f"Entity {i}: Offset SAI!")

# Report
if errors:
    print("\n❌ VALIDATION FAILED!")
    for err in errors:
        print(f"  - {err}")
    raise ValueError("Validation errors detected!")
else:
    print("\n✓ ALL CHECKS PASSED!")

print("="*60)

In [ ]:
# Cell 10: Xuất JSON
import json

# Prepare output
result = {
    'text': TEXT,
    'entities': [
        {
            'text': ent['text'],
            'type': ent['type'],
            'start': ent['start'],
            'end': ent['end'],
            'score': ent['score'],
            'source': ent['source']
        }
        for ent in final_entities
    ]
}

# Save
output_path = '/kaggle/working/ner_output.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"✓ Saved to {output_path}")
print(f"\nTotal entities: {len(final_entities)}")
print(f"\nType breakdown:")
for t, count in sorted(type_counts.items()):
    print(f"  {t}: {count}")

# Pretty print first 20
print(f"\n{'='*60}")
print("First 20 entities:")
print(f"{'='*60}")
for i, ent in enumerate(final_entities[:20]):
    print(f"{i+1:2d}. [{ent['type']:20s}] '{ent['text']}'")

print(f"\n{'='*60}")
print("✓ INFERENCE COMPLETE!")
print(f"{'='*60}")